# Merge, crop, and drift-correct 3i SlideBook time-lapse movies

1. **Merge channels** - SlideBook (3i) exports one TIFF per channel (`..._Z0_T0000_C0.tif`, `..._C1.tif`, ...). Draw one or more crop boxes; each crop is saved as its own ImageJ `TCYX` hyperstack (`..._crop1_..._merged.tif`) with all channels, in a folder you choose. Cropping keeps each movie small enough to load into memory for analysis.
2. **Drift correction** - streaming XY drift correction of every merged movie. The selected channel is used to estimate drift, and the same shift is applied to every channel and Z plane at each timepoint.

Run the cells from top to bottom. If you already have merged hyperstacks, skip section 1 and set `input_paths` in section 2 directly.

## Requirement

This notebook uses `tifffile`. If the import below fails, run `%pip install tifffile` once in a new cell, then restart the kernel.


In [ ]:
from __future__ import annotations

import csv
import glob
import os
import re
import shutil
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import tifffile

## 1. Merge single-channel TIFFs into one hyperstack per crop

Files named `<prefix>_Z#_T#_C#.tif` in the source folder are grouped by `<prefix>` (one group per field of view / XY position). Channel names, pixel size, and frame interval are read from the matching SlideBook `.log` file and saved in the merged TIFFs. Channels keep their SlideBook order (C0 -> channel 0, C1 -> channel 1, ...).

The merge reads one frame at a time, so it never holds a whole channel in memory. Indexing each channel TIFF over a network share can take ~20-30 s; the readers are kept open and reused by the cells below.

In [ ]:
CHANNEL_FILE_PATTERN = re.compile(
    r"^(?P<prefix>.+)_Z(?P<z>\d+)_T(?P<t>\d+)_C(?P<c>\d+)\.tiff?$",
    re.IGNORECASE,
)


@dataclass(frozen=True)
class ChannelSet:
    """One field of view exported by SlideBook as one TIFF per channel."""

    prefix: str
    folder: Path
    channel_paths: tuple[Path, ...]
    channel_names: tuple[str, ...]
    log_path: Path | None
    microns_per_pixel: float | None
    frame_interval_s: float | None


def _parse_slidebook_log(log_path: Path | None) -> dict[str, object]:
    """Read pixel size, frame interval, and channel names from a SlideBook .log file."""
    parsed: dict[str, object] = {"channel_names_by_file": {}}
    if log_path is None or not log_path.is_file():
        return parsed
    lines = log_path.read_text(errors="replace").splitlines()
    for line in lines:
        if line.startswith("Microns Per Pixel:"):
            parsed["microns_per_pixel"] = float(line.split(":", 1)[1])
        elif line.startswith("Average Timelapse Interval:"):
            match = re.search(r"([\d.]+)\s*ms", line)
            if match:
                parsed["frame_interval_s"] = float(match.group(1)) / 1000.0
        elif line.startswith("Time Points:"):
            parsed["timepoints"] = int(line.split(":", 1)[1])
        elif line.startswith("Channels:"):
            parsed["channels"] = int(line.split(":", 1)[1])
    header_index = next(
        (index for index, line in enumerate(lines) if line.startswith("IFD\t")), None
    )
    if header_index is not None:
        header = lines[header_index].split("\t")
        if "Channel Name" in header and "TIFF File Name" in header:
            name_column = header.index("Channel Name")
            file_column = header.index("TIFF File Name")
            for line in lines[header_index + 1 :]:
                fields = line.split("\t")
                if len(fields) > max(name_column, file_column):
                    parsed["channel_names_by_file"][fields[file_column].strip()] = (
                        fields[name_column].strip()
                    )
    return parsed


def find_channel_sets(folder: str | os.PathLike[str]) -> list[ChannelSet]:
    """Group single-channel TIFFs named <prefix>_Z#_T#_C#.tif by field of view."""
    folder = Path(folder)
    if not folder.is_dir():
        raise FileNotFoundError(f"Folder not found: {folder}")

    grouped: dict[str, dict[int, Path]] = {}
    for path in sorted(folder.iterdir()):
        match = CHANNEL_FILE_PATTERN.match(path.name)
        if not match or not path.is_file():
            continue
        prefix = match.group("prefix")
        channel = int(match.group("c"))
        if channel in grouped.setdefault(prefix, {}):
            raise ValueError(
                f"Two files claim channel {channel} for {prefix}: "
                f"{grouped[prefix][channel].name} and {path.name}"
            )
        grouped[prefix][channel] = path

    channel_sets = []
    for prefix, by_channel in sorted(grouped.items()):
        indices = sorted(by_channel)
        if indices != list(range(len(indices))):
            raise ValueError(f"{prefix}: expected channels C0..C{len(indices) - 1}, found {indices}.")
        paths = tuple(by_channel[index] for index in indices)
        logs = sorted(folder.glob(f"{glob.escape(prefix)}_*.log"))
        log_path = logs[0] if logs else None
        log = _parse_slidebook_log(log_path)
        if "channels" in log and log["channels"] != len(paths):
            raise ValueError(
                f"{prefix}: the log lists {log['channels']} channels but "
                f"{len(paths)} channel TIFFs were found."
            )
        names = tuple(
            log["channel_names_by_file"].get(path.name, f"Channel {index}")
            for index, path in enumerate(paths)
        )
        channel_sets.append(
            ChannelSet(
                prefix=prefix,
                folder=folder,
                channel_paths=paths,
                channel_names=names,
                log_path=log_path,
                microns_per_pixel=log.get("microns_per_pixel"),
                frame_interval_s=log.get("frame_interval_s"),
            )
        )
    return channel_sets


class ChannelReader:
    """Read one frame at a time from a single-channel (frames, Y, X) TIFF.

    SlideBook interleaves the TIFF headers with the pixel data, so these files
    cannot be memory-mapped. Reading page by page keeps memory use to one frame.
    """

    def __init__(self, path: str | os.PathLike[str]):
        self.path = Path(path)
        self._tif = tifffile.TiffFile(self.path)
        try:
            series = self._tif.series[0]
            shape = tuple(int(v) for v in series.shape)
            if len(shape) == 2:
                shape = (1, *shape)
            if len(shape) != 3 or series.axes[-2:].upper() != "YX":
                raise ValueError(
                    f"{self.path.name}: expected a single-channel (frames, Y, X) TIFF, "
                    f"found axes={series.axes!r}, shape={series.shape}."
                )
            self._pages = series.pages
            if len(self._pages) != shape[0]:
                raise ValueError(f"{self.path.name}: {shape[0]} frames but {len(self._pages)} TIFF pages.")
            self.shape = shape
            self.dtype = np.dtype(series.dtype)
        except Exception:
            self._tif.close()
            raise

    def frame(self, t: int) -> np.ndarray:
        return np.asarray(self._pages[t].asarray())

    def close(self) -> None:
        self._tif.close()


# Opening a channel TIFF takes ~20-30 s over the network because every page header
# is indexed, so readers are cached and reused by the preview and merge functions.
_OPEN_CHANNEL_READERS: dict[Path, ChannelReader] = {}


def channel_readers(channel_set: ChannelSet, show_progress: bool = True) -> list[ChannelReader]:
    readers = []
    for index, path in enumerate(channel_set.channel_paths):
        if path not in _OPEN_CHANNEL_READERS:
            if show_progress:
                print(f"indexing C{index}: {path.name}", flush=True)
            _OPEN_CHANNEL_READERS[path] = ChannelReader(path)
        readers.append(_OPEN_CHANNEL_READERS[path])
    return readers


def close_channel_readers() -> None:
    while _OPEN_CHANNEL_READERS:
        _, reader = _OPEN_CHANNEL_READERS.popitem()
        reader.close()


def _physical_memory_bytes() -> int | None:
    try:
        if os.name == "nt":
            import ctypes

            class MEMORYSTATUSEX(ctypes.Structure):
                _fields_ = [
                    ("dwLength", ctypes.c_ulong),
                    ("dwMemoryLoad", ctypes.c_ulong),
                    ("ullTotalPhys", ctypes.c_ulonglong),
                    ("ullAvailPhys", ctypes.c_ulonglong),
                    ("ullTotalPageFile", ctypes.c_ulonglong),
                    ("ullAvailPageFile", ctypes.c_ulonglong),
                    ("ullTotalVirtual", ctypes.c_ulonglong),
                    ("ullAvailVirtual", ctypes.c_ulonglong),
                    ("ullAvailExtendedVirtual", ctypes.c_ulonglong),
                ]

            status = MEMORYSTATUSEX()
            status.dwLength = ctypes.sizeof(MEMORYSTATUSEX)
            ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(status))
            return int(status.ullTotalPhys)
        return int(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES"))
    except Exception:
        return None


def _memory_limit_gb(max_memory_gb: float | None) -> float:
    """None = half of this PC's RAM; 0 = no limit."""
    if max_memory_gb is None:
        ram = _physical_memory_bytes()
        return 0.5 * ram / 1e9 if ram else 0.0
    return float(max_memory_gb)


def _as_crop_list(crops) -> list[tuple[int, int, int, int]] | None:
    """Accept None, one (y0, y1, x0, x1) tuple, or a list of them."""
    if crops is None:
        return None
    crops = list(crops)
    if len(crops) == 4 and all(isinstance(v, (int, np.integer)) for v in crops):
        crops = [crops]
    return [tuple(int(v) for v in crop) for crop in crops]


def _normalize_crops(crops, height: int, width: int) -> list[tuple[int, int, int, int]]:
    crop_list = _as_crop_list(crops)
    if crop_list is None:
        return [(0, height, 0, width)]
    if not crop_list:
        raise ValueError("crops is empty; use None for the full field.")
    for number, crop in enumerate(crop_list, 1):
        if len(crop) != 4:
            raise ValueError(f"crop #{number} = {crop} must be (y0, y1, x0, x1).")
        y0, y1, x0, x1 = crop
        if not (0 <= y0 < y1 <= height and 0 <= x0 < x1 <= width):
            raise ValueError(
                f"crop #{number} = {crop} must satisfy 0 <= y0 < y1 <= {height} and 0 <= x0 < x1 <= {width}."
            )
    if len(set(crop_list)) != len(crop_list):
        raise ValueError(f"crops contains duplicates: {crop_list}")
    return crop_list


def describe_channel_set(
    channel_set: ChannelSet,
    frames: range | None = None,
    crops=None,
    max_memory_gb: float | None = None,
    show_progress: bool = True,
) -> dict[str, object]:
    """Check that all channels match, validate frames/crops, and report each output's size.

    crops: None (full field), one (y0, y1, x0, x1) tuple, or a list of them.
    """
    readers = channel_readers(channel_set, show_progress=show_progress)
    if len({(r.shape, r.dtype) for r in readers}) != 1:
        details = ", ".join(f"C{i}: {r.shape} {r.dtype}" for i, r in enumerate(readers))
        raise ValueError(f"{channel_set.prefix}: channel TIFFs do not match ({details}).")
    total_frames, height, width = readers[0].shape
    dtype = readers[0].dtype

    frames = range(total_frames) if frames is None else frames
    if not isinstance(frames, range) or len(frames) == 0 or frames.step <= 0:
        raise ValueError("frames must be None or a non-empty increasing range, e.g. range(0, 300).")
    if frames.start < 0 or frames[-1] >= total_frames:
        raise ValueError(f"frames={frames} is outside 0..{total_frames - 1}.")
    crop_list = _normalize_crops(crops, height, width)
    limit = _memory_limit_gb(max_memory_gb)

    crop_rows = []
    for number, (y0, y1, x0, x1) in enumerate(crop_list, 1):
        merged_shape = (len(frames), len(readers), y1 - y0, x1 - x0)
        pixels = int(np.prod(merged_shape, dtype=np.int64))
        open_gb = round(pixels * 4 / 1e9, 2)  # RAM to hold the movie as float32, as tracking tools do
        crop_rows.append(
            {
                "crop": number,
                "crop_yx": (y0, y1, x0, x1),
                "merged_shape_TCYX": merged_shape,
                "merged_file_GB": round(pixels * dtype.itemsize / 1e9, 2),
                "memory_to_open_GB": open_gb,
                "over_limit": bool(limit) and open_gb > limit,
            }
        )
    ram = _physical_memory_bytes()
    return {
        "prefix": channel_set.prefix,
        "channels": {f"C{i}": name for i, name in enumerate(channel_set.channel_names)},
        "source_frames_Y_X": (total_frames, height, width),
        "dtype": str(dtype),
        "frames": frames,
        "crops": crop_rows,
        "memory_limit_GB": round(limit, 1),
        "this_pc_RAM_GB": None if ram is None else round(ram / 1e9, 1),
        "microns_per_pixel": channel_set.microns_per_pixel,
        "frame_interval_s": channel_set.frame_interval_s,
    }


def choose_folder(
    title: str = "Choose a folder",
    initial_dir: str | os.PathLike[str] | None = None,
) -> Path:
    """Open a folder dialog (it may appear behind the browser) and return the chosen folder."""
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    try:
        chosen = filedialog.askdirectory(
            parent=root,
            title=title,
            initialdir=str(initial_dir) if initial_dir else None,
            mustexist=False,
        )
    finally:
        root.destroy()
    if not chosen:
        raise RuntimeError("No folder chosen. Rerun the cell, or type the folder path instead.")
    return Path(chosen)


def default_merged_paths(
    channel_set: ChannelSet,
    crops=None,
    output_dir: str | os.PathLike[str] | None = None,
) -> list[Path]:
    """One output per crop: <prefix>_crop<N>_y<y0>-<y1>_x<x0>-<x1>_merged.tif.

    crops=None (full field) gives <prefix>_merged.tif.
    """
    output_dir = channel_set.folder / "merged" if output_dir is None else Path(output_dir)
    crop_list = _as_crop_list(crops)
    if crop_list is None:
        return [output_dir / f"{channel_set.prefix}_merged.tif"]
    return [
        output_dir / f"{channel_set.prefix}_crop{number}_y{y0}-{y1}_x{x0}-{x1}_merged.tif"
        for number, (y0, y1, x0, x1) in enumerate(crop_list, 1)
    ]


def show_channel_preview(channel_set: ChannelSet, frame: int = 0, crops=None) -> None:
    """Show one frame of every channel, with each crop outlined and numbered."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle

    readers = channel_readers(channel_set)
    crop_list = _as_crop_list(crops) or []
    fig, axes = plt.subplots(1, len(readers), figsize=(4.5 * len(readers), 4.8), squeeze=False)
    for c, (ax, reader, name) in enumerate(zip(axes[0], readers, channel_set.channel_names)):
        image = reader.frame(frame).astype(np.float32)
        lo, hi = np.percentile(image, [1, 99.8])
        ax.imshow(image, cmap="gray", vmin=lo, vmax=hi)
        ax.set_title(f"C{c}: {name} (frame {frame})")
        for number, (y0, y1, x0, x1) in enumerate(crop_list, 1):
            ax.add_patch(Rectangle((x0 - 0.5, y0 - 0.5), x1 - x0, y1 - y0, fill=False, edgecolor="yellow", linewidth=1.2))
            ax.text(x0 + 15, y0 + 15, f"#{number}", color="yellow", fontsize=9, va="top")
    fig.tight_layout()
    plt.show()


ROI_CHANNEL_COLORS = ("lime", "red", "magenta", "cyan", "yellow", "dodgerblue")


def select_crops(
    channel_set: ChannelSet,
    initial_crops=None,
    channels_shown: list[int] | None = None,
    projection_frames: int = 10,
    box_size: tuple[int, int] | None = None,
    merge_frames: range | None = None,
    max_memory_gb: float | None = None,
    channel_colors: tuple[str, ...] = ROI_CHANNEL_COLORS,
) -> list[tuple[int, int, int, int]] | None:
    """Draw one or more crop boxes with the mouse in a pop-up window.

    Returns a list of (y0, y1, x0, x1); each crop becomes its own merged TIFF.

    Display: every channel is a colored max projection of `projection_frames` frames
    spread over the movie, blended additively, so overlapping signal mixes colors.
    Tick/untick the channel checkboxes (or press 1-9) to show or hide channels.

    Drawing: drag to draw a box (drag inside it to move it, or its handles to resize),
    then press A / "Add crop" to keep it. D / "Delete last" removes the last crop.
    Enter, "Done", or closing the window accepts (a drawn-but-not-added box is added).
    Esc cancels and returns initial_crops. box_size=(height, width) fixes the box size.
    """
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.colors import to_rgb
    from matplotlib.patches import Rectangle
    from matplotlib.widgets import Button, CheckButtons, RectangleSelector

    readers = channel_readers(channel_set)
    total_frames, height, width = readers[0].shape
    starting_crops = [] if initial_crops is None else _normalize_crops(initial_crops, height, width)
    sampled = np.unique(np.linspace(0, total_frames - 1, max(1, projection_frames)).round().astype(int))
    print(f"building max projections of {len(sampled)} frames x {len(readers)} channels...", flush=True)
    normalized = []  # per channel, max projection scaled to 0..1
    for reader in readers:
        projection = None
        for t in sampled:
            plane = reader.frame(int(t)).astype(np.float32)
            projection = plane if projection is None else np.maximum(projection, plane)
        lo, hi = np.percentile(projection, [1, 99.8])
        normalized.append(np.clip((projection - lo) / max(hi - lo, 1e-6), 0, 1))
    colors = [np.array(to_rgb(channel_colors[c % len(channel_colors)]), dtype=np.float32) for c in range(len(readers))]
    shown = [True] * len(readers) if channels_shown is None else [c in channels_shown for c in range(len(readers))]

    n_frames = total_frames if merge_frames is None else len(merge_frames)
    limit = _memory_limit_gb(max_memory_gb)

    def crop_gb(crop) -> float:
        y0, y1, x0, x1 = crop
        return n_frames * len(readers) * (y1 - y0) * (x1 - x0) * 4 / 1e9

    def composite() -> np.ndarray:
        rgb = np.zeros((height, width, 3), dtype=np.float32)
        for on, image, color in zip(shown, normalized, colors):
            if on:
                rgb += image[..., None] * color
        return np.clip(rgb, 0, 1)

    def to_crop(extents) -> tuple[int, int, int, int]:
        # Pixel i spans i-0.5..i+0.5 in imshow coordinates.
        xmin, xmax, ymin, ymax = extents
        x0, x1 = (int(np.clip(np.floor(v + 0.5), 0, width)) for v in (xmin, xmax))
        y0, y1 = (int(np.clip(np.floor(v + 0.5), 0, height)) for v in (ymin, ymax))
        return y0, y1, x0, x1

    def to_extents(crop) -> tuple[float, float, float, float]:
        y0, y1, x0, x1 = crop
        return x0 - 0.5, x1 - 0.5, y0 - 0.5, y1 - 0.5

    def fixed_size(crop) -> tuple[int, int, int, int]:
        box_h, box_w = min(box_size[0], height), min(box_size[1], width)
        y0, y1, x0, x1 = crop
        cy, cx = (y0 + y1) / 2, (x0 + x1) / 2
        y0 = int(np.clip(round(cy - box_h / 2), 0, height - box_h))
        x0 = int(np.clip(round(cx - box_w / 2), 0, width - box_w))
        return y0, y0 + box_h, x0, x0 + box_w

    state = {"crops": list(starting_crops), "current": None, "cancelled": False}
    crop_artists: list = []

    previous_backend = matplotlib.get_backend()
    plt.switch_backend("TkAgg")
    fig = None
    try:
        fig = plt.figure(figsize=(12, 9))
        ax = fig.add_axes([0.05, 0.06, 0.66, 0.80])
        shown_image = ax.imshow(composite())
        ax.set_xlabel("x (pixels)")
        ax.set_ylabel("y (pixels)")
        fig.suptitle(
            f"{channel_set.prefix} - max of {len(sampled)} frames\n"
            "drag = draw box | A = add crop | D = delete last | 1-9 = toggle channel | "
            "Enter/close = done | Esc = cancel",
            fontsize=10,
        )

        panel_x, panel_w = 0.74, 0.24
        check_h = 0.045 * len(readers)
        check_top = 0.86
        check_ax = fig.add_axes([panel_x, check_top - check_h, panel_w, check_h])
        check_ax.set_title("channels shown", fontsize=10, loc="left")
        labels = [f"C{c}: {name}" for c, name in enumerate(channel_set.channel_names)]
        try:
            checks = CheckButtons(
                check_ax, labels, shown,
                label_props={"color": [tuple(color) for color in colors], "fontsize": [11] * len(labels)},
                frame_props={"edgecolor": "white", "facecolor": "white", "s": 160},
                check_props={"facecolor": "black", "s": 160},
            )
        except TypeError:  # matplotlib < 3.7
            checks = CheckButtons(check_ax, labels, shown)
        check_ax.set_facecolor("0.25")

        button_top = check_top - check_h - 0.03
        add_button = Button(fig.add_axes([panel_x, button_top - 0.05, panel_w, 0.045]), "Add crop (A)")
        delete_button = Button(fig.add_axes([panel_x, button_top - 0.105, panel_w, 0.045]), "Delete last (D)")
        done_button = Button(fig.add_axes([panel_x, button_top - 0.16, panel_w, 0.045]), "Done (Enter)")
        list_text = fig.text(panel_x, button_top - 0.19, "", fontsize=9, va="top", family="monospace")

        def update_title(message: str | None = None) -> None:
            current = state["current"]
            if message is not None:
                ax.set_title(message, color="black", fontsize=10)
            elif current is None:
                ax.set_title("drag to draw a box, then press A (Add crop)", color="black", fontsize=10)
            else:
                y0, y1, x0, x1 = current
                gb = crop_gb(current)
                over = bool(limit) and gb > limit
                ax.set_title(
                    f"new box ({y0}, {y1}, {x0}, {x1})  {y1 - y0} x {x1 - x0} px  "
                    f"RAM to open ~{gb:.1f} GB for {n_frames} frames"
                    + (f" (over {limit:.1f} GB limit)" if over else "")
                    + "  - press A to add",
                    color="red" if over else "black",
                    fontsize=10,
                )
            fig.canvas.draw_idle()

        def redraw_crops() -> None:
            for artist in crop_artists:
                artist.remove()
            crop_artists.clear()
            lines = []
            for number, crop in enumerate(state["crops"], 1):
                y0, y1, x0, x1 = crop
                gb = crop_gb(crop)
                over = bool(limit) and gb > limit
                color = "red" if over else "orange"
                crop_artists.append(
                    ax.add_patch(Rectangle(to_extents(crop)[::2], x1 - x0, y1 - y0, fill=False, edgecolor=color, linewidth=1.8))
                )
                crop_artists.append(
                    ax.text(x0 + 15, y0 + 15, f"#{number}", color=color, fontsize=12, fontweight="bold", va="top")
                )
                lines.append(
                    f"#{number} ({y0}, {y1}, {x0}, {x1})\n   {y1 - y0} x {x1 - x0} px, ~{gb:.1f} GB"
                    + ("  OVER LIMIT" if over else "")
                )
            list_text.set_text(
                f"crops (y0, y1, x0, x1)\nRAM limit {limit:.1f} GB each\n\n"
                + ("\n".join(lines) if lines else "none yet")
            )
            update_title()

        def on_select(_press=None, _release=None) -> None:
            crop = to_crop(selector.extents)
            if box_size is not None:
                crop = fixed_size(crop)
                selector.extents = to_extents(crop)
            if crop[1] - crop[0] >= 2 and crop[3] - crop[2] >= 2:
                state["current"] = crop
            update_title()

        def clear_selector() -> None:
            try:
                selector.clear()
            except AttributeError:
                selector.set_visible(False)

        def add_current(_event=None) -> None:
            current = state["current"]
            if current is None:
                update_title("draw a box first, then press A")
                return
            if current not in state["crops"]:
                state["crops"].append(current)
            state["current"] = None
            clear_selector()
            redraw_crops()

        def delete_last(_event=None) -> None:
            if state["crops"]:
                state["crops"].pop()
            redraw_crops()

        def done(_event=None) -> None:
            plt.close(fig)

        def on_check(_label) -> None:
            shown[:] = list(checks.get_status())
            shown_image.set_data(composite())
            fig.canvas.draw_idle()

        def on_key(event) -> None:
            key = (event.key or "").lower()
            if key == "enter":
                done()
            elif key == "escape":
                state["cancelled"] = True
                plt.close(fig)
            elif key == "a":
                add_current()
            elif key in ("d", "delete"):
                delete_last()
            elif key.isdigit() and 1 <= int(key) <= len(readers):
                checks.set_active(int(key) - 1)  # toggles; on_check redraws

        selector = RectangleSelector(
            ax,
            on_select,
            useblit=True,
            button=[1],
            minspanx=2,
            minspany=2,
            spancoords="data",
            interactive=True,
            drag_from_anywhere=True,
            props=dict(edgecolor="yellow", facecolor="yellow", alpha=0.15, fill=True),
        )
        checks.on_clicked(on_check)
        add_button.on_clicked(add_current)
        delete_button.on_clicked(delete_last)
        done_button.on_clicked(done)
        fig.canvas.mpl_connect("key_press_event", on_key)
        # Keep references while the window is open (widgets without one stop responding).
        fig._crop_widgets = (selector, checks, add_button, delete_button, done_button)
        fig._crop_selector, fig._crop_checks, fig._crop_on_select = selector, checks, on_select

        if box_size is not None:
            state["current"] = fixed_size((height // 2, height // 2, width // 2, width // 2))
            selector.extents = to_extents(state["current"])
        redraw_crops()
        try:  # bring the window in front of the browser
            window = fig.canvas.manager.window
            window.attributes("-topmost", 1)
            window.after(500, lambda: window.attributes("-topmost", 0))
        except Exception:
            pass
        print("A crop window is open (it may be behind other windows). Draw boxes, press A after each, then Enter.", flush=True)
        plt.show(block=True)
    finally:
        if fig is not None:
            plt.close(fig)
        plt.switch_backend(previous_backend)

    if state["cancelled"]:
        print(f"Cancelled; crops stay {initial_crops}")
        return initial_crops
    crops = list(state["crops"])
    if state["current"] is not None and state["current"] not in crops:
        crops.append(state["current"])
        print("(the last box was drawn but not added; it was added)")
    if not crops:
        print(f"No crops drawn; crops stay {initial_crops}")
        return initial_crops
    print("crops = [")
    for crop in crops:
        print(f"    {crop},")
    print("]")
    return crops


def merge_channel_tiffs(
    channel_set: ChannelSet,
    crops=None,
    output_dir: str | os.PathLike[str] | None = None,
    frames: range | None = None,
    max_memory_gb: float | None = None,
    overwrite: bool = False,
    show_progress: bool = True,
) -> list[dict[str, object]]:
    """Stream single-channel TIFFs into one ImageJ TCYX hyperstack per crop.

    The source files are read once, frame by frame; every crop is cut from each frame
    and written to its own file at the same time, so extra crops add little time.
    crops: None (full field), one (y0, y1, x0, x1) tuple, or a list of them.
    frames: range of zero-based source frames to keep (None = all).
    max_memory_gb: stop before writing if opening any one output as float32
        (as tracking tools do) would need more RAM than this (None = 50% of this PC's RAM; 0 disables the check).
    """
    import queue
    import threading
    import time

    info = describe_channel_set(
        channel_set, frames=frames, crops=crops,
        max_memory_gb=max_memory_gb, show_progress=show_progress,
    )
    frames = info["frames"]
    crop_rows = info["crops"]
    dtype = np.dtype(info["dtype"])
    outputs = default_merged_paths(channel_set, crops, output_dir)
    limit = info["memory_limit_GB"]

    too_big = [row for row in crop_rows if row["over_limit"]]
    if too_big:
        details = "\n".join(
            f"  crop #{row['crop']} {row['crop_yx']}: ~{row['memory_to_open_GB']} GB" for row in too_big
        )
        raise MemoryError(
            f"Opening these as float32 would need more than the {limit} GB RAM limit:\n{details}\n"
            "Choose smaller crops and/or fewer frames, or set max_memory_gb=0 "
            "to merge anyway (e.g. for Fiji or a larger-memory PC)."
        )
    existing = [path for path in outputs if path.exists()]
    if existing and not overwrite:
        raise FileExistsError(
            "Output already exists:\n" + "\n".join(f"  {path}" for path in existing)
            + "\nSet overwrite=True to replace."
        )
    sources = {path.resolve() for path in channel_set.channel_paths}
    if any(path.resolve() in sources for path in outputs):
        raise ValueError("Outputs must differ from the channel TIFFs.")
    outputs[0].parent.mkdir(parents=True, exist_ok=True)
    sizes = [int(np.prod(row["merged_shape_TCYX"], dtype=np.int64)) * dtype.itemsize for row in crop_rows]
    free_bytes = shutil.disk_usage(outputs[0].parent).free
    if free_bytes < sum(sizes) * 1.05:
        raise OSError(
            f"Not enough space in {outputs[0].parent}: need ~{sum(sizes) / 1e9:.1f} GB, "
            f"{free_bytes / 1e9:.1f} GB free."
        )

    readers = channel_readers(channel_set, show_progress=show_progress)
    if show_progress:
        print(
            "Merging channels\n"
            + "".join(
                f"  C{i} ({name}): {path.name}\n"
                for i, (name, path) in enumerate(zip(channel_set.channel_names, channel_set.channel_paths))
            )
            + f"  frames: {frames}\n"
            + "".join(
                f"  crop #{row['crop']} {row['crop_yx']} -> {path.name}  "
                f"TCYX {row['merged_shape_TCYX']} ~{row['merged_file_GB']} GB\n"
                for row, path in zip(crop_rows, outputs)
            )
            + f"  folder: {outputs[0].parent}",
            flush=True,
        )

    metadata: dict[str, object] = {"axes": "TCYX"}
    resolution = None
    if channel_set.microns_per_pixel:
        metadata["unit"] = "um"
        resolution = (1.0 / channel_set.microns_per_pixel, 1.0 / channel_set.microns_per_pixel)
    if channel_set.frame_interval_s:
        metadata["finterval"] = channel_set.frame_interval_s * frames.step

    partials = [path.with_name(f".{path.name}.{os.getpid()}.partial.tif") for path in outputs]
    queues = [queue.Queue(maxsize=4 * len(readers)) for _ in outputs]
    errors: list[BaseException | None] = [None] * len(outputs)

    def write_one(index: int) -> None:
        plane_queue = queues[index]

        def planes():
            while True:
                plane = plane_queue.get()
                if plane is None:
                    return
                yield plane

        try:
            tifffile.imwrite(
                partials[index],
                data=planes(),
                shape=crop_rows[index]["merged_shape_TCYX"],
                dtype=dtype,
                imagej=True,
                bigtiff=sizes[index] >= 2**32 - 2**25,
                photometric="minisblack",
                metadata=metadata,
                resolution=resolution,
            )
        except BaseException as exc:
            errors[index] = exc
            while plane_queue.get() is not None:  # keep the reader from blocking
                pass

    try:
        writers = [threading.Thread(target=write_one, args=(i,), daemon=True) for i in range(len(outputs))]
        for writer in writers:
            writer.start()
        start = time.time()
        try:
            for count, t in enumerate(frames):
                if any(error is not None for error in errors):
                    break
                for reader in readers:
                    plane = reader.frame(t)
                    for plane_queue, row in zip(queues, crop_rows):
                        y0, y1, x0, x1 = row["crop_yx"]
                        plane_queue.put(np.ascontiguousarray(plane[y0:y1, x0:x1]))
                if show_progress and (count == 0 or (count + 1) % 50 == 0 or count + 1 == len(frames)):
                    elapsed = time.time() - start
                    remaining = elapsed / (count + 1) * (len(frames) - count - 1)
                    print(
                        f"merged frame {count + 1}/{len(frames)}  "
                        f"({elapsed / 60:.1f} min elapsed, ~{remaining / 60:.1f} min left)",
                        flush=True,
                    )
        finally:
            for plane_queue in queues:
                plane_queue.put(None)
            for writer in writers:
                writer.join()
        for path, error in zip(outputs, errors):
            if error is not None:
                raise RuntimeError(f"Writing {path.name} failed: {error!r}") from error

        # Spot-check planes of every output against the source files before publishing.
        check_indices = sorted({0, len(frames) // 2, len(frames) - 1})
        written = [tifffile.TiffFile(path) for path in partials]
        try:
            for tif, row in zip(written, crop_rows):
                series = tif.series[0]
                if series.axes != "TCYX" or tuple(series.shape) != row["merged_shape_TCYX"]:
                    raise RuntimeError(
                        f"crop #{row['crop']} reads back as {series.axes} {series.shape}, "
                        f"expected TCYX {row['merged_shape_TCYX']}."
                    )
            for index in check_indices:
                for c, reader in enumerate(readers):
                    source = reader.frame(frames[index])
                    for tif, row in zip(written, crop_rows):
                        y0, y1, x0, x1 = row["crop_yx"]
                        page = tif.series[0].pages[index * len(readers) + c].asarray()
                        if not np.array_equal(page, source[y0:y1, x0:x1]):
                            raise RuntimeError(f"Pixel check failed: crop #{row['crop']}, T={index}, C={c}.")
        finally:
            for tif in written:
                tif.close()
        for partial, path in zip(partials, outputs):
            os.replace(partial, path)
    finally:
        for partial in partials:
            if partial.exists():
                partial.unlink()

    results = [
        {**row, "frames": str(frames), "output_path": str(path)}
        for row, path in zip(crop_rows, outputs)
    ]
    if show_progress:
        print("Done (pixels spot-checked against the source files)")
        for row in results:
            print(f"  crop #{row['crop']}: {row['output_path']}  (~{row['memory_to_open_GB']} GB RAM to open)")
    return results

### Choose the position, crops, and frames

**Size matters.** Tracking and analysis tools typically load the entire movie into memory as 32-bit floats. For example, a 2096 x 2152 field with 4 channels is ~72 MB per frame, so 1350 frames would need ~97 GB of RAM. Crop to the cell(s) you want to analyze and/or keep a frame range. The merge stops before writing if opening any one crop would need more than half of this PC's RAM (`max_memory_gb=0` turns this off).

`crops` is a list of boxes `(y0, y1, x0, x1)` in full-frame pixels (row range, then column range). Each box becomes its own merged TIFF. Type them here, or draw them with the mouse in the next cell. `crops = None` merges the full field. `merge_frames` is a zero-based `range`, e.g. `range(0, 600)` or `range(0, 1350, 2)` (every other frame; the saved frame interval is adjusted).

In [ ]:
source_folder = Path(r"\\server\share\SlideBook_export\experiment")  # folder with the per-channel TIFFs and .log files

channel_sets = find_channel_sets(source_folder)
for index, found in enumerate(channel_sets):
    names = ", ".join(f"C{c}={name}" for c, name in enumerate(found.channel_names))
    print(f"[{index}] {found.prefix}: {names}")

position_index = 0  # which position (group) above to merge
channel_set = channel_sets[position_index]

# List of (y0, y1, x0, x1) boxes, one merged TIFF each; None = full field.
# Example: one 512 x 512 px box. Or draw the boxes in the next cell.
crops = [
    (850, 1362, 200, 712),
]
merge_frames = None  # None = all frames; e.g. range(0, 600)

describe_channel_set(channel_set, frames=merge_frames, crops=crops)

### Draw the crop boxes (optional)

Opens a separate window (it may appear behind the browser). Each channel is shown as a colored max projection of `projection_frames` frames spread across the movie, so you can see where cells move over time. The crops already in `crops` are drawn in orange and numbered.

- **Channels:** tick/untick the checkboxes on the right (or press **1-4**) to show or hide each channel. Shown channels add together, so overlapping signal mixes colors: C0 lime + C2 magenta = white, C0 lime + C1 red = yellow, C2 magenta + C3 cyan = light blue/white. `channels_shown=[0, 2]` starts with only those ticked; `channel_colors=` changes the colors.
- **Drawing:** drag on the image to draw a box (drag inside it to move it, or its handles to resize it), then press **A** or **Add crop** to keep it. Repeat for more crops. **D** or **Delete last** removes the last crop.
- The panel lists each crop and the RAM needed to open it; crops over the limit are shown in **red**.
- Press **Enter**, click **Done**, or close the window to accept (a box drawn but not yet added is added). Press **Esc** to cancel and keep the previous `crops`.
- `box_size=(512, 512)` keeps every new box at a fixed size, so you only position it.

The window is drawn with Tk (included with Anaconda/Python), so this needs Jupyter running on this PC. Skip this cell to use the `crops` typed above.

In [ ]:
crops = select_crops(
    channel_set,
    initial_crops=crops,
    channels_shown=None,  # channels ticked at start; None = all, e.g. [0, 2]
    projection_frames=10,  # frames in each channel's max projection
    box_size=None,  # e.g. (512, 512) for fixed-size boxes
    merge_frames=merge_frames,
)
describe_channel_set(channel_set, frames=merge_frames, crops=crops)

### Preview (optional)

Shows one frame of each channel with the numbered crop boxes, to check them in every channel before merging.

In [ ]:
%matplotlib inline
show_channel_preview(channel_set, frame=0, crops=crops)

### Choose where to save

The merged crop TIFFs are saved in `merged_output_dir`; the drift-corrected TIFFs and shift CSVs from section 2 are saved beside them. A local folder (e.g. under Documents) makes drift correction and later loading faster than a network share. The merge checks there is enough free space before writing.

By default a folder dialog opens (it may appear behind the browser); you can create a new folder in it. To skip the dialog, type the folder instead, e.g. `merged_output_dir = Path(r"C:\Users\me\Documents\my_experiment")`.

In [ ]:
merged_output_dir = choose_folder(
    title="Where to save the merged crop TIFFs",
    initial_dir=Path.home() / "Documents",
)
# merged_output_dir = Path(r"C:\path\to\local\folder")  # or type it instead of the dialog

merged_output_dir.mkdir(parents=True, exist_ok=True)
merged_paths = default_merged_paths(channel_set, crops, merged_output_dir)
print(f"Saving to: {merged_output_dir}  ({shutil.disk_usage(merged_output_dir).free / 1e9:.1f} GB free)")
for path in merged_paths:
    print(f"  {path.name}{'  (already exists)' if path.exists() else ''}")

### Run the merge

Writes one `<prefix>_crop<N>_y<y0>-<y1>_x<x0>-<x1>_merged.tif` per crop (ImageJ `TCYX` hyperstack, same bit depth, no rescaling) and spot-checks pixels against the source files. The source TIFFs are not modified. The source files are read only once for all crops: reading every frame of the 4 channels from the server takes on the order of 30 min, whatever the number or size of the crops.

In [ ]:
merge_results = merge_channel_tiffs(
    channel_set,
    crops=crops,
    output_dir=merged_output_dir,
    frames=merge_frames,
    max_memory_gb=None,  # None = half of this PC's RAM; 0 = no limit
    overwrite=False,
    show_progress=True,
)
close_channel_readers()
merged_paths = [Path(result["output_path"]) for result in merge_results]

## 2. Drift correction

In [ ]:
@dataclass(frozen=True)
class TiffStackInfo:
    path: Path
    series_index: int
    axes: str
    shape: tuple[int, ...]
    dtype: np.dtype
    page_count: int
    timepoints: int
    channels: int
    z_slices: int
    height: int
    width: int
    is_imagej: bool


def default_output_paths(
    input_path: str | os.PathLike[str],
    output_dir: str | os.PathLike[str] | None = None,
) -> tuple[Path, Path]:
    """Default outputs: <stem>_drift_corrected.tif and <stem>_drift_shifts.csv.

    They go in output_dir, or the current working directory when it is None.
    """
    source = Path(input_path)
    folder = Path.cwd() if output_dir is None else Path(output_dir)
    output_path = folder / f"{source.stem}_drift_corrected.tif"
    shifts_csv = folder / f"{source.stem}_drift_shifts.csv"
    return output_path, shifts_csv


def _read_tiff_info_from_open_file(
    tif: tifffile.TiffFile,
    path: Path,
    series_index: int,
) -> TiffStackInfo:
    if not 0 <= series_index < len(tif.series):
        raise ValueError(
            f"series_index={series_index} is invalid; this TIFF has {len(tif.series)} series."
        )
    series = tif.series[series_index]
    axes = series.axes.upper()
    shape = tuple(int(value) for value in series.shape)
    if len(axes) != len(shape) or not axes.endswith("YX"):
        raise ValueError(f"Expected a TIFF series ending in YX, found axes={axes!r}, shape={shape}.")
    if len(set(axes)) != len(axes):
        raise ValueError(f"Repeated axis labels are not supported: axes={axes!r}.")

    unsupported = [
        (axis, size)
        for axis, size in zip(axes, shape)
        if axis not in "TCZYX" and size != 1
    ]
    if unsupported:
        raise ValueError(
            "Only T, C, Z, Y, and X axes are supported. "
            f"Non-singleton unsupported axes: {unsupported}."
        )

    leading_shape = shape[:-2]
    expected_pages = int(np.prod(leading_shape, dtype=np.int64)) if leading_shape else 1
    if len(series.pages) != expected_pages:
        raise ValueError(
            "This notebook expects one 2D TIFF page per T/C/Z plane, but "
            f"axes={axes!r}, shape={shape} implies {expected_pages} pages and "
            f"the series exposes {len(series.pages)}."
        )

    sizes = dict(zip(axes, shape))
    return TiffStackInfo(
        path=path,
        series_index=series_index,
        axes=axes,
        shape=shape,
        dtype=np.dtype(series.dtype),
        page_count=len(series.pages),
        timepoints=int(sizes.get("T", 1)),
        channels=int(sizes.get("C", 1)),
        z_slices=int(sizes.get("Z", 1)),
        height=int(sizes["Y"]),
        width=int(sizes["X"]),
        is_imagej=bool(tif.is_imagej),
    )


def read_tiff_info(
    input_path: str | os.PathLike[str],
    series_index: int = 0,
) -> TiffStackInfo:
    path = Path(input_path)
    if not path.is_file():
        raise FileNotFoundError(f"TIFF not found: {path}")
    with tifffile.TiffFile(path) as tif:
        return _read_tiff_info_from_open_file(tif, path, series_index)


def inspect_tiff(
    input_path: str | os.PathLike[str],
    series_index: int = 0,
    output_dir: str | os.PathLike[str] | None = None,
) -> dict[str, object]:
    info = read_tiff_info(input_path, series_index=series_index)
    output_path, shifts_csv = default_output_paths(input_path, output_dir=output_dir)
    return {
        "input_path": str(info.path),
        "series_index": info.series_index,
        "axes": info.axes,
        "shape": info.shape,
        "dtype": str(info.dtype),
        "pages": info.page_count,
        "timepoints": info.timepoints,
        "channels": info.channels,
        "z_slices": info.z_slices,
        "height": info.height,
        "width": info.width,
        "is_imagej": info.is_imagej,
        "default_output_path": str(output_path),
        "default_shifts_csv": str(shifts_csv),
    }


def _plane_index(info: TiffStackInfo, t: int, c: int, z: int) -> int:
    values = {"T": t, "C": c, "Z": z}
    coordinates = tuple(values.get(axis, 0) for axis in info.axes[:-2])
    if not coordinates:
        return 0
    return int(np.ravel_multi_index(coordinates, info.shape[:-2]))


def _read_plane(
    series: tifffile.TiffPageSeries,
    info: TiffStackInfo,
    t: int,
    c: int,
    z: int,
) -> np.ndarray:
    return np.asarray(series.pages[_plane_index(info, t, c, z)].asarray())


def robust_normalize(image: np.ndarray) -> np.ndarray:
    image = np.asarray(image, dtype=np.float32)
    lo, hi = np.percentile(image, [1, 99])
    if hi <= lo:
        centered = image - np.mean(image)
    else:
        centered = np.clip((image - lo) / (hi - lo), 0, 1)
        centered -= np.mean(centered)
    return centered


def phase_correlation_shift(
    reference: np.ndarray,
    moving: np.ndarray,
) -> tuple[float, float]:
    """Return the subpixel shift (dy, dx) to apply to moving."""
    ref = robust_normalize(reference)
    mov = robust_normalize(moving)
    if ref.shape != mov.shape:
        raise ValueError(f"Image shapes differ: reference={ref.shape}, moving={mov.shape}.")

    window_y = np.hanning(ref.shape[0]) if ref.shape[0] > 1 else np.ones(1)
    window_x = np.hanning(ref.shape[1]) if ref.shape[1] > 1 else np.ones(1)
    window = np.outer(window_y, window_x).astype(np.float32)
    ref *= window
    mov *= window

    cross_power = np.fft.fftn(ref) * np.conj(np.fft.fftn(mov))
    cross_power /= np.maximum(np.abs(cross_power), 1e-12)
    correlation = np.fft.ifftn(cross_power).real
    peak = np.unravel_index(np.argmax(correlation), correlation.shape)
    shift = np.array(peak, dtype=np.float64)
    for dimension, size in enumerate(ref.shape):
        if shift[dimension] > size // 2:
            shift[dimension] -= size

    refined = shift.copy()
    for dimension, size in enumerate(ref.shape):
        index = list(peak)
        index[dimension] = (peak[dimension] - 1) % size
        previous = correlation[tuple(index)]
        index[dimension] = peak[dimension]
        center = correlation[tuple(index)]
        index[dimension] = (peak[dimension] + 1) % size
        following = correlation[tuple(index)]
        denominator = previous - 2 * center + following
        if abs(denominator) > 1e-12:
            refined[dimension] += 0.5 * (previous - following) / denominator
    return float(refined[0]), float(refined[1])


def fourier_shift_2d(image: np.ndarray, dy: float, dx: float) -> np.ndarray:
    image_float = np.asarray(image, dtype=np.float32)
    ny, nx = image_float.shape
    fy = np.fft.fftfreq(ny)[:, None]
    fx = np.fft.fftfreq(nx)[None, :]
    phase = np.exp(-2j * np.pi * (dy * fy + dx * fx))
    shifted = np.fft.ifft2(np.fft.fft2(image_float) * phase).real
    return shifted.astype(np.float32, copy=False)


def cast_corrected_plane(plane: np.ndarray, dtype: np.dtype) -> np.ndarray:
    dtype = np.dtype(dtype)
    if np.issubdtype(dtype, np.integer):
        limits = np.iinfo(dtype)
        plane = np.clip(np.rint(plane), limits.min, limits.max)
    return plane.astype(dtype, copy=False)


def _timepoint_projection(
    series: tifffile.TiffPageSeries,
    info: TiffStackInfo,
    time_index: int,
    registration_channel: int,
) -> np.ndarray:
    projection = None
    for z in range(info.z_slices):
        plane = _read_plane(series, info, time_index, registration_channel, z)
        plane = np.asarray(plane, dtype=np.float32)
        projection = plane if projection is None else np.maximum(projection, plane)
    if projection is None:
        raise RuntimeError("No planes were available for registration.")
    return projection


def estimate_tiff_shifts(
    input_path: str | os.PathLike[str],
    series_index: int = 0,
    reference_time: int = 0,
    registration_channel: int = 0,
    show_progress: bool = True,
) -> tuple[TiffStackInfo, list[tuple[int, float, float]]]:
    path = Path(input_path)
    with tifffile.TiffFile(path) as tif:
        info = _read_tiff_info_from_open_file(tif, path, series_index)
        if not 0 <= reference_time < info.timepoints:
            raise ValueError(f"reference_time must be between 0 and {info.timepoints - 1}.")
        if not 0 <= registration_channel < info.channels:
            raise ValueError(
                f"registration_channel must be between 0 and {info.channels - 1}."
            )

        series = tif.series[series_index]
        reference = _timepoint_projection(
            series, info, reference_time, registration_channel
        )
        shifts: list[tuple[int, float, float]] = []
        for t in range(info.timepoints):
            moving = _timepoint_projection(series, info, t, registration_channel)
            dy, dx = phase_correlation_shift(reference, moving)
            shifts.append((t, dy, dx))
            if show_progress and (
                t == 0 or (t + 1) % 25 == 0 or t + 1 == info.timepoints
            ):
                print(
                    f"estimating shift {t + 1}/{info.timepoints}: "
                    f"dy={dy:.3f}, dx={dx:.3f}",
                    flush=True,
                )
    return info, shifts


def _imagej_metadata(tif: tifffile.TiffFile, axes: str) -> dict[str, object]:
    # Tifffile regenerates structural ImageJ fields (images/channels/slices/frames).
    # Retain useful acquisition/display metadata but omit fields tied to old pixels.
    source = dict(tif.imagej_metadata or {})
    for key in (
        "ImageJ",
        "images",
        "channels",
        "slices",
        "frames",
        "hyperstack",
        "min",
        "max",
        "Ranges",
    ):
        source.pop(key, None)
    source["axes"] = axes
    return source


def write_corrected_tiff(
    input_path: str | os.PathLike[str],
    output_path: str | os.PathLike[str],
    info: TiffStackInfo,
    shifts: list[tuple[int, float, float]],
    show_progress: bool = True,
) -> None:
    input_path = Path(input_path)
    output_path = Path(output_path)
    if input_path.resolve() == output_path.resolve():
        raise ValueError("output_path must differ from input_path.")
    if len(shifts) != info.timepoints:
        raise ValueError(
            f"Expected {info.timepoints} shifts, received {len(shifts)}."
        )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    partial_path = output_path.with_name(
        f".{output_path.name}.{os.getpid()}.partial.tif"
    )

    with tifffile.TiffFile(input_path) as tif:
        series = tif.series[info.series_index]
        metadata = _imagej_metadata(tif, info.axes)
        first_page = series.pages[0]
        x_resolution = first_page.tags.get("XResolution")
        y_resolution = first_page.tags.get("YResolution")
        resolution = None
        if x_resolution is not None and y_resolution is not None:
            resolution = (x_resolution.value, y_resolution.value)
        resolution_unit = first_page.tags.get("ResolutionUnit")
        resolution_unit = resolution_unit.value if resolution_unit is not None else None

        def corrected_planes():
            leading_shape = info.shape[:-2]
            for page_number, page in enumerate(series.pages):
                coordinates = (
                    np.unravel_index(page_number, leading_shape)
                    if leading_shape
                    else ()
                )
                coordinate_by_axis = dict(zip(info.axes[:-2], coordinates))
                time_index = int(coordinate_by_axis.get("T", 0))
                _, dy, dx = shifts[time_index]
                shifted = fourier_shift_2d(page.asarray(), dy, dx)
                yield cast_corrected_plane(shifted, info.dtype)
                if show_progress and (
                    page_number == 0
                    or (page_number + 1) % 100 == 0
                    or page_number + 1 == info.page_count
                ):
                    print(
                        f"writing page {page_number + 1}/{info.page_count}",
                        flush=True,
                    )

        estimated_bytes = int(np.prod(info.shape, dtype=np.int64)) * info.dtype.itemsize
        try:
            tifffile.imwrite(
                partial_path,
                data=corrected_planes(),
                shape=info.shape,
                dtype=info.dtype,
                imagej=True,
                bigtiff=estimated_bytes >= 2**32 - 2**25,
                photometric="minisblack",
                metadata=metadata,
                resolution=resolution,
                resolutionunit=resolution_unit,
            )
            os.replace(partial_path, output_path)
        finally:
            if partial_path.exists():
                partial_path.unlink()


def _write_shifts_csv(
    shifts_csv: str | os.PathLike[str],
    shifts: list[tuple[int, float, float]],
) -> None:
    path = Path(shifts_csv)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["time_index", "shift_y_pixels", "shift_x_pixels"])
        writer.writerows(shifts)


def drift_correct_tiff(
    input_path: str | os.PathLike[str],
    output_path: str | os.PathLike[str] | None = None,
    shifts_csv: str | os.PathLike[str] | None = None,
    series_index: int = 0,
    reference_time: int = 0,
    registration_channel: int = 0,
    show_progress: bool = True,
) -> dict[str, object]:
    input_path = Path(input_path)
    default_output, default_csv = default_output_paths(input_path)
    output_path = default_output if output_path is None else Path(output_path)
    shifts_csv = default_csv if shifts_csv is None else Path(shifts_csv)

    info = read_tiff_info(input_path, series_index=series_index)
    if show_progress:
        print(
            "TIFF drift correction\n"
            f"  input: {input_path}\n"
            f"  output: {output_path}\n"
            f"  shifts CSV: {shifts_csv}\n"
            f"  axes/shape: {info.axes} {info.shape}\n"
            f"  dtype: {info.dtype}\n"
            f"  registration channel (zero-based): {registration_channel}\n"
            f"  reference time (zero-based): {reference_time}"
        )

    info, shifts = estimate_tiff_shifts(
        input_path=input_path,
        series_index=series_index,
        reference_time=reference_time,
        registration_channel=registration_channel,
        show_progress=show_progress,
    )
    write_corrected_tiff(
        input_path=input_path,
        output_path=output_path,
        info=info,
        shifts=shifts,
        show_progress=show_progress,
    )
    _write_shifts_csv(shifts_csv, shifts)

    check = read_tiff_info(output_path)
    if check.shape != info.shape or check.axes != info.axes or check.dtype != info.dtype:
        raise RuntimeError(
            "Output verification failed: "
            f"input {info.axes} {info.shape} {info.dtype}, "
            f"output {check.axes} {check.shape} {check.dtype}."
        )

    result = {
        "input_path": str(input_path),
        "output_path": str(output_path),
        "shifts_csv": str(shifts_csv),
        "axes": info.axes,
        "shape": info.shape,
        "dtype": str(info.dtype),
        "shifts": shifts,
        "final_shift_y_pixels": float(shifts[-1][1]) if shifts else 0.0,
        "final_shift_x_pixels": float(shifts[-1][2]) if shifts else 0.0,
    }
    if show_progress:
        print(
            "Done\n"
            f"  output: {result['output_path']}\n"
            f"  shifts CSV: {result['shifts_csv']}\n"
            f"  verified axes/shape/dtype: {check.axes} {check.shape} {check.dtype}"
        )
    return result

### Choose inputs and settings

`input_paths` defaults to the merged crop TIFFs from section 1 (every crop is corrected separately, since each crop can drift differently). To correct other hyperstacks instead, list their paths. Each corrected TIFF and shift CSV is written beside its input. `registration_channel=0` means the first channel; both channel and time indices are zero-based.

In [ ]:
input_paths = [Path(path) for path in merged_paths]
# input_paths = [Path(r"C:\path\to\movie_merged.tif")]  # or list files yourself

# Both indices are zero-based: 0 = first channel / first timepoint.
series_index = 0
reference_time = 0
registration_channel = 0

for path in input_paths:
    info = read_tiff_info(path, series_index=series_index)
    corrected, shifts = default_output_paths(path, output_dir=path.parent)
    print(f"{path.name}: {info.axes} {info.shape} {info.dtype}")
    print(f"  -> {corrected.name}\n  -> {shifts.name}")

### Run drift correction

For each input this writes a new ImageJ-compatible TIFF and a CSV containing the estimated shifts. Inputs are never overwritten, and the outputs keep the input's `TCYX` shape, channels, and bit depth.

In [ ]:
results = []
for path in input_paths:
    corrected, shifts = default_output_paths(path, output_dir=path.parent)
    results.append(
        drift_correct_tiff(
            input_path=path,
            output_path=corrected,
            shifts_csv=shifts,
            series_index=series_index,
            reference_time=reference_time,
            registration_channel=registration_channel,
            show_progress=True,
        )
    )
[result["output_path"] for result in results]

### View shifts

In [ ]:
try:
    import pandas as pd

    for result in results:
        print(Path(result["input_path"]).name)
        display(pd.read_csv(result["shifts_csv"]))
except ImportError:
    for result in results:
        with Path(result["shifts_csv"]).open(newline="") as file:
            print(Path(result["input_path"]).name, list(csv.DictReader(file))[:10])

### Plot drift

In [ ]:
try:
    import matplotlib.pyplot as plt
    import pandas as pd

    fig, axes = plt.subplots(len(results), 1, figsize=(8, 3.5 * len(results)), squeeze=False)
    for ax, result in zip(axes[:, 0], results):
        shifts = pd.read_csv(result["shifts_csv"])
        shifts.plot(x="time_index", y=["shift_y_pixels", "shift_x_pixels"], marker="o", grid=True, ax=ax)
        ax.set_xlabel("Time index")
        ax.set_ylabel("Shift to apply (pixels)")
        ax.set_title(f"Estimated XY drift - {Path(result['input_path']).name}", fontsize=9)
    fig.tight_layout()
    plt.show()
except ImportError as exc:
    print(f"Install pandas/matplotlib to plot shifts: {exc}")